In [1]:

# ============================================
# PHASE 1 - STEP 1: ENVIRONMENT INITIALIZATION
# ============================================

import sys
import os
import platform

print("CONSUMER COMPLAINT RESOLUTION ASSISTANT")
print("=" * 45)

print("Python Version:", sys.version)
print("Operating System:", platform.system())

try:
    import torch

    print("PyTorch Version:", torch.__version__)

    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
    else:
        print("GPU: Not available (CPU mode)")

except ImportError:
    print("PyTorch: Not installed")

print("=" * 45)
print("Environment check completed!")


CONSUMER COMPLAINT RESOLUTION ASSISTANT
Python Version: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Operating System: Linux
PyTorch Version: 2.14.1+cu130
GPU: Tesla T4
Environment check completed!


In [2]:
# Install required libraries

!pip install -q \
    pymupdf \
    sentence-transformers \
    faiss-cpu \
    rank-bm25 \
    langgraph \
    gradio \
    plotly \
    pandas \
    numpy \
    scikit-learn

print("All required libraries installed successfully!")

All required libraries installed successfully!


In [3]:
import fitz
import torch
import faiss
import gradio
import plotly
import pandas
import numpy
import sklearn

from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from langgraph.graph import StateGraph, START, END

print("LIBRARY VERIFICATION")
print("=" * 40)
print("PyMuPDF:        Ready")
print("PyTorch:        Ready")
print("FAISS:          Ready")
print("Sentence Transformers: Ready")
print("BM25:           Ready")
print("LangGraph:      Ready")
print("Gradio:         Ready")
print("Plotly:         Ready")
print("Pandas:         Ready")
print("NumPy:          Ready")
print("Scikit-learn:   Ready")
print("=" * 40)
print("All libraries imported successfully!")

/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


LIBRARY VERIFICATION
PyMuPDF:        Ready
PyTorch:        Ready
FAISS:          Ready
Sentence Transformers: Ready
BM25:           Ready
LangGraph:      Ready
Gradio:         Ready
Plotly:         Ready
Pandas:         Ready
NumPy:          Ready
Scikit-learn:   Ready
All libraries imported successfully!


In [4]:
#Step 5 – Upload PDFs and Extract Text using PyMuPDF

!pip install -q pymupdf pandas


In [5]:

from google.colab import files

uploaded = files.upload()


Saving Consumer_Protection_Documents.zip to Consumer_Protection_Documents.zip


In [6]:

import zipfile
import os

zip_path = next(
    name for name in uploaded
    if name.lower().endswith(".zip")
)

extract_folder = "/content/consumer_documents"
os.makedirs(extract_folder, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_folder)

print("ZIP extraction completed!")
print("Folder:", extract_folder)


ZIP extraction completed!
Folder: /content/consumer_documents


In [7]:

import os

pdf_files = []

for root, dirs, filenames in os.walk(extract_folder):
    for filename in filenames:
        if filename.lower().endswith(".pdf"):
            pdf_files.append(os.path.join(root, filename))

print("Total PDF documents found:", len(pdf_files))

for i, path in enumerate(pdf_files, 1):
    print(f"{i}. {os.path.basename(path)}")


Total PDF documents found: 35
1. Provisions of Act comes into force.pdf
2. The National Consumer Disputes Redressal Commission (Group ‘C’ posts) Recruitment Rules, 2022.pdf
3. The Consumer Protection Act, 2019.pdf
4. The Central Consumer Protection Authority (Allocation and Transaction of Business) Regulations, 2020.pdf
5. The Consumer Protection (Consumer Disputes Redressal Commissions) (Amendment) Rules, 2023.pdf
6. Corrigendum to CP Act 2019.pdf
7. The Guidelines for Prevention and Regulation of Dark Patterns, 2023.pdf
8. The Consumer Protection (Administrative Control over The State Commission and The District Commission) Regulations, 202.pdf
9. Corrigendum to The Consumer Protection (E-Commerce) Rules, 2020.pdf
10. The Consumer Protection (Consumer Disputes Redressal Commissions) Amendment Rules, 2022.pdf
11. The Consumer Protection (Central Consumer Protection Council) Rules, 2020.pdf
12. The Consumer Protection (E-Commerce) (Amendment) Rules, 2026.pdf
13. The Guidelines for Prev

In [8]:

import fitz
import pandas as pd

documents = []

for pdf_path in pdf_files:
    pdf = fitz.open(pdf_path)

    for page_num, page in enumerate(pdf, start=1):
        text = page.get_text("text").strip()

        if text:
            documents.append({
                "document_name": os.path.basename(pdf_path),
                "page_number": page_num,
                "text": text
            })

    pdf.close()

df = pd.DataFrame(documents)

print("Text extraction completed!")
print("Total extracted pages:", len(df))
print("Total documents:", df["document_name"].nunique())

df.head()


Text extraction completed!
Total extracted pages: 318
Total documents: 33


,document_name,page_number,text
0,Provisions of Act comes into force.pdf,1,3259 GI/2020 \n(1) \nरजिस्ट्री सं. डी.एल.- 330...
1,Provisions of Act comes into force.pdf,2,2 \nTHE GAZETTE OF INDIA : EXTRAORDINARY \n [...
2,The National Consumer Disputes Redressal Commi...,1,4387 GI/2022 \n(1) \nरिज᭭ᮝी सं. डी.एल.- 33004/...
3,The National Consumer Disputes Redressal Commi...,2,2 \nTHE GAZETTE OF INDIA : EXTRAORDINARY \n ...
4,The National Consumer Disputes Redressal Commi...,3,[भागII—ख᭛ ड 3(i)]\nभारत का राजपᮢ : असाधारण \n3...


In [9]:

import re

def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[ \t]+', ' ', text)
    return text.strip()

df["clean_text"] = df["text"].apply(clean_text)

print("Text cleaning completed!")
print("Total pages:", len(df))
print("\nSample cleaned text:")
print(df["clean_text"].iloc[0][:500])


Text cleaning completed!
Total pages: 318

Sample cleaned text:
3259 GI/2020 (1) रजिस्ट्री सं. डी.एल.- 33004/99 REGD. No. D. L.-33004/99 xxxGIDHxxx xxxGIDExxx असाधारण EXTRAORDINARY भाग II—खण् ड 3—उप-खण् ड (ii) PART II—Section 3—Sub-section (ii) प्राजधकार से प्रकाजित PUBLISHED BY AUTHORITY उपभोक्ता मामले, खाद्य और सार्विजिक जर्तरण मंत्रालय (उपभोक्ता मामले जर्भाग) अजधसूचिा िई दिल्ली, 23 िुलाई, 2020 dk-vk- 2421¼v½-&&केंद्रीय सरकार, उपभोक्ता संरक्षण अजधजियम, 2019 (2019 का 35) की धारा 1 की उपधारा (3) द्वारा प्रित्त िजक्तयों का प्रयोग करते हुए, 24 िुलाई, 2020 को उ


In [10]:

def create_chunks(text, chunk_size=1000, overlap=150):
    chunks = []
    start = 0

    while start < len(text):
        end = min(start + chunk_size, len(text))

        if end < len(text):
            split_at = text.rfind(" ", start, end)
            if split_at > start:
                end = split_at

        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        if end >= len(text):
            break

        start = end - overlap

    return chunks


chunk_records = []

for _, row in df.iterrows():
    page_chunks = create_chunks(row["clean_text"])

    for chunk_id, chunk in enumerate(page_chunks):
        chunk_records.append({
            "document_name": row["document_name"],
            "page_number": row["page_number"],
            "chunk_id": chunk_id,
            "text": chunk
        })

chunks_df = pd.DataFrame(chunk_records)

print("Chunking completed!")
print("Total chunks:", len(chunks_df))
print("Total documents:", chunks_df["document_name"].nunique())


Chunking completed!
Total chunks: 1011
Total documents: 33


In [11]:

print(chunks_df.head(5).to_string(index=False))

print("\nChunk statistics:")
print(chunks_df["text"].str.len().describe())


                                                                                    document_name  page_number  chunk_id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                

In [12]:

chunks_df.to_csv(
    "/content/consumer_knowledge_base.csv",
    index=False
)

print("Knowledge base saved successfully!")


Knowledge base saved successfully!


In [13]:

!pip install -q sentence-transformers faiss-cpu


In [14]:

from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "paraphrase-multilingual-MiniLM-L12-v2"
)

print("Embedding model loaded successfully!")


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (4 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!


In [16]:

import numpy as np

texts = chunks_df["text"].tolist()

embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

embeddings = embeddings.astype("float32")

print("Embedding generation completed!")
print("Embedding shape:", embeddings.shape)
print("Total chunks:", len(texts))
print("Vector dimensions:", embeddings.shape[1])


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Embedding generation completed!
Embedding shape: (1011, 384)
Total chunks: 1011
Vector dimensions: 384


In [17]:

import faiss

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(embeddings)

print("FAISS vector database created!")
print("Total vectors stored:", index.ntotal)
print("Vector dimensions:", dimension)


FAISS vector database created!
Total vectors stored: 1011
Vector dimensions: 384


In [18]:

query = "What are the rights of consumers in India?"

query_embedding = model.encode(
    [query],
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

scores, indices = index.search(query_embedding, k=5)

print("Query:", query)
print("\nTop 5 relevant results:\n")

for rank, (idx, score) in enumerate(
    zip(indices[0], scores[0]), start=1
):
    result = chunks_df.iloc[idx]

    print(f"Result {rank}")
    print("Document:", result["document_name"])
    print("Page:", result["page_number"])
    print("Similarity:", round(float(score), 4))
    print("Text:", result["text"][:400])
    print("-" * 60)


Query: What are the rights of consumers in India?

Top 5 relevant results:

Result 1
Document: The Consumer Protection Act, 2019.pdf
Page: 4
Similarity: 0.7307
Text: 4 THE GAZETTE OF INDIA EXTRAORDINARY [PART II— (iii) the right to be assured, wherever possible, access to a variety of goods, products or services at competitive prices; (iv) the right to be heard and to be assured that consumer's interests will receive due consideration at appropriate fora; (v) the right to seek redressal against unfair trade practice or restrictive trade practices or unscrupulo
------------------------------------------------------------
Result 2
Document: The Guidelines for Prevention and Regulation of Dark Patterns, 2023.pdf
Page: 2
Similarity: 0.7093
Text: 2 THE GAZETTE OF INDIA : EXTRAORDINARY [PART III—SEC.4] (ग) “केन्द्रीय उपभोक्ता संरक्षण प्राजधकरण” से उपभोक्ता संरक्षण अजधजनयम 2019 (2019 का 35) की धारा 10 के अधीन स्ट्र्ाजपत प्राजधकरण अजभप्रेत है; (घ) “वाजणजययक लाभ” से दकसी भी प्रकार का मौदरक लाभ 

In [19]:

faiss.write_index(
    index,
    "/content/consumer_faiss.index"
)

np.save(
    "/content/consumer_embeddings.npy",
    embeddings
)

chunks_df.to_pickle(
    "/content/consumer_chunks.pkl"
)

print("FAISS database saved successfully!")
print("Embeddings saved successfully!")
print("Chunk metadata saved successfully!")


FAISS database saved successfully!
Embeddings saved successfully!
Chunk metadata saved successfully!


In [20]:

!pip install -q rank-bm25


In [21]:

import re
from rank_bm25 import BM25Okapi

def tokenize(text):
    return re.findall(r"\w+", str(text).lower(), flags=re.UNICODE)

tokenized_corpus = [
    tokenize(text) for text in chunks_df["text"].tolist()
]

bm25 = BM25Okapi(tokenized_corpus)

print("BM25 database created successfully!")
print("Total documents indexed:", len(tokenized_corpus))


BM25 database created successfully!
Total documents indexed: 1011


In [22]:

query = "consumer rights under Consumer Protection Act 2019"

query_tokens = tokenize(query)

bm25_scores = bm25.get_scores(query_tokens)

top_k = 5
bm25_indices = np.argsort(bm25_scores)[::-1][:top_k]

print("BM25 Search Results:\n")

for rank, idx in enumerate(bm25_indices, start=1):
    result = chunks_df.iloc[idx]

    print(f"Result {rank}")
    print("Document:", result["document_name"])
    print("Page:", result["page_number"])
    print("BM25 Score:", round(float(bm25_scores[idx]), 4))
    print("Text:", result["text"][:350])
    print("-" * 60)


BM25 Search Results:

Result 1
Document: The Guidelines for Prevention and Regulation of Dark Patterns, 2023.pdf
Page: 7
BM25 Score: 16.8046
Text: 2019), the Central Consumer Protection Authority hereby issues the following guidelines to provide for the prevention and regulation of dark patterns, namely:- 1. Short title and commencement. —(1) These guidelines may be called the Guidelines for Prevention and Regulation of Dark Patterns, 2023 (2) It shall come into force on the date of its publi
------------------------------------------------------------
Result 2
Document: The Central Consumer Protection Authority (Allocation and Transaction of Business) Regulations, 2020.pdf
Page: 3
BM25 Score: 16.8027
Text: 14 of the Consumer Protection Act, 2019 (35 of 2019), the Central Consumer Protection Authority, with the previous approval of the Central Government, hereby makes the following regulations, namely:-- 1. Short title and commencement. --- (1) These regulations may be called the Centr

In [23]:

def hybrid_search(query, top_k=5, candidate_k=20, rrf_k=60):

    # 1. FAISS semantic search
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    dense_scores, dense_indices = index.search(
        query_embedding, candidate_k
    )

    # 2. BM25 keyword search
    query_tokens = tokenize(query)
    sparse_scores = bm25.get_scores(query_tokens)

    sparse_indices = np.argsort(sparse_scores)[::-1][:candidate_k]

    # 3. Reciprocal Rank Fusion
    rrf_scores = {}

    for rank, idx in enumerate(dense_indices[0], start=1):
        if idx >= 0:
            rrf_scores[int(idx)] = (
                rrf_scores.get(int(idx), 0) +
                1 / (rrf_k + rank)
            )

    for rank, idx in enumerate(sparse_indices, start=1):
        rrf_scores[int(idx)] = (
            rrf_scores.get(int(idx), 0) +
            1 / (rrf_k + rank)
        )

    # 4. Final ranking
    ranked_indices = sorted(
        rrf_scores,
        key=rrf_scores.get,
        reverse=True
    )[:top_k]

    results = []

    for idx in ranked_indices:
        row = chunks_df.iloc[idx]

        results.append({
            "document_name": row["document_name"],
            "page_number": row["page_number"],
            "text": row["text"],
            "rrf_score": rrf_scores[idx]
        })

    return results


print("Hybrid retrieval function created!")


Hybrid retrieval function created!


In [24]:

test_query = "What are the rights of consumers in India?"

hybrid_results = hybrid_search(test_query, top_k=5)

print("HYBRID RETRIEVAL RESULTS")
print("=" * 70)

for rank, result in enumerate(hybrid_results, start=1):

    print(f"\nResult {rank}")
    print("Document:", result["document_name"])
    print("Page:", result["page_number"])
    print("RRF Score:", round(result["rrf_score"], 5))
    print("Text:", result["text"][:400])
    print("-" * 70)


HYBRID RETRIEVAL RESULTS

Result 1
Document: The Consumer Protection Act, 2019.pdf
Page: 13
RRF Score: 0.02951
Text: ovided for the protection of consumers under any other law for the time being in force and recommend appropriate remedial measures for their effective implementation; (e) recommend adoption of international covenants and best international practices on consumer rights to ensure effective enforcement of consumer rights; (f) undertake and promote research in the field of consumer rights; (g) spread 
----------------------------------------------------------------------

Result 2
Document: The Consumer Protection (Direct Selling) Rules, 2021.pdf
Page: 14
RRF Score: 0.02942
Text: 14 THE GAZETTE OF INDIA : EXTRAORDINARY [PART II—SEC. 3(i)] (h) direct selling shall not state or imply that a guarantee, warranty or other expression having substantially the same meaning, offers to the consumer any rights in additional to those provided by law, when it does not; (i) the terms of a

In [ ]:

import pickle

with open("/content/consumer_bm25.pkl", "wb") as f:
    pickle.dump(bm25, f)

with open("/content/consumer_tokenized_corpus.pkl", "wb") as f:
    pickle.dump(tokenized_corpus, f)

print("BM25 database saved successfully!")
print("Hybrid retrieval is ready!")


BM25 database saved successfully!
Hybrid retrieval is ready!


In [ ]:
#Step 9 – Query Expansion and Improved Retrieval

def expand_query(query):

    q = query.lower()

    variations = []

    if "consumer" in q and (
        "right" in q or "protection" in q
    ):
        variations = [
            "What are the six consumer rights under Indian law?",
            "Right to safety and protection against hazardous goods",
            "Right to be informed about quality quantity and price",
            "Right to choose goods and services",
            "Right to be heard and seek consumer redressal",
            "Right to consumer education",
            "Consumer rights under Consumer Protection Act 2019"
        ]

    elif "refund" in q or "return" in q:
        variations = [
            "Consumer rights regarding refunds",
            "Return and replacement of defective products",
            "Refund policy under consumer protection law",
            "Compensation for defective goods",
            "Unfair trade practices related to refunds"
        ]

    elif "online shopping" in q or "e-commerce" in q:
        variations = [
            "E-commerce consumer protection rules",
            "Online shopping complaints and refunds",
            "Responsibilities of e-commerce platforms",
            "Consumer rights against online sellers"
        ]

    elif "complaint" in q or "grievance" in q:
        variations = [
            "Consumer complaint filing procedure",
            "Consumer dispute redressal process",
            "District Consumer Commission complaint",
            "Consumer grievance resolution"
        ]

    return list(dict.fromkeys([query] + variations))


query = "What are the rights of consumers in India?"

expanded_queries = expand_query(query)

print("Original Query:", query)
print("\nExpanded Queries:")

for i, q in enumerate(expanded_queries, 1):
    print(i, q)

print("\nTotal queries:", len(expanded_queries))


Original Query: What are the rights of consumers in India?

Expanded Queries:
1 What are the rights of consumers in India?
2 What are the six consumer rights under Indian law?
3 Right to safety and protection against hazardous goods
4 Right to be informed about quality quantity and price
5 Right to choose goods and services
6 Right to be heard and seek consumer redressal
7 Right to consumer education
8 Consumer rights under Consumer Protection Act 2019

Total queries: 8


In [ ]:

def expanded_hybrid_search(query, top_k=5):

    queries = expand_query(query)

    combined_scores = {}
    result_data = {}

    for expanded_query in queries:

        results = hybrid_search(
            expanded_query,
            top_k=10
        )

        for rank, result in enumerate(results, start=1):

            key = (
                result["document_name"],
                result["page_number"],
                result["text"]
            )

            combined_scores[key] = (
                combined_scores.get(key, 0) +
                1 / (60 + rank)
            )

            result_data[key] = result

    ranked_keys = sorted(
        combined_scores,
        key=combined_scores.get,
        reverse=True
    )[:top_k]

    final_results = []

    for key in ranked_keys:
        result = result_data[key].copy()
        result["expanded_score"] = combined_scores[key]
        final_results.append(result)

    return final_results


expanded_results = expanded_hybrid_search(
    "What are the rights of consumers in India?",
    top_k=5
)

print("QUERY EXPANSION RESULTS")
print("=" * 65)

for rank, result in enumerate(expanded_results, 1):

    print("\nResult:", rank)
    print("Document:", result["document_name"])
    print("Page:", result["page_number"])
    print("Expanded Score:", round(result["expanded_score"], 5))
    print("Text:", result["text"][:400])
    print("-" * 65)


QUERY EXPANSION RESULTS

Result: 1
Document: The Consumer Protection Act, 2019.pdf
Page: 3
Expanded Score: 0.11213
Text: s been made, denies or disputes the allegations contained in the complaint; (9) "consumer rights" includes,— (i) the right to be protected against the marketing of goods, products or services which are hazardous to life and property; (ii) the right to be informed about the quality, quantity, potency, purity, standard and price of goods, products or services, as the case may be, so as to protect th
-----------------------------------------------------------------

Result: 2
Document: The Consumer Protection Act, 2019.pdf
Page: 4
Expanded Score: 0.10999
Text: 4 THE GAZETTE OF INDIA EXTRAORDINARY [PART II— (iii) the right to be assured, wherever possible, access to a variety of goods, products or services at competitive prices; (iv) the right to be heard and to be assured that consumer's interests will receive due consideration at appropriate fora; (v) the right to seek

In [ ]:
#Step 10 – Cross-Encoder Reranking
!pip install -q sentence-transformers

In [ ]:
from sentence_transformers import CrossEncoder

# Load multilingual Cross-Encoder model
reranker = CrossEncoder(
    "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"
)

print("Cross-Encoder model loaded successfully!")

config.json:   0%|          | 0.00/891 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Cross-Encoder model loaded successfully!


In [ ]:
# Step 10.3 - Cross-Encoder Reranking

def cross_encoder_rerank(query, results, top_k=5):

    # Create query-passage pairs
    pairs = [
        [query, result["text"]]
        for result in results
    ]

    # Predict relevance scores
    scores = reranker.predict(pairs)

    # Add scores to results
    reranked_results = []

    for result, score in zip(results, scores):
        item = result.copy()
        item["rerank_score"] = float(score)
        reranked_results.append(item)

    # Sort by Cross-Encoder score
    reranked_results.sort(
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    return reranked_results[:top_k]


print("Cross-Encoder reranking function created successfully!")

Cross-Encoder reranking function created successfully!


In [ ]:
# Step 10.4 - Test Cross-Encoder Reranking

query = "What are the rights of consumers in India?"

# Retrieve expanded hybrid search results
candidate_results = expanded_hybrid_search(query, top_k=10)

# Rerank using Cross-Encoder
reranked_results = cross_encoder_rerank(
    query,
    candidate_results,
    top_k=5
)

# Display results
print("CROSS-ENCODER RERANKING RESULTS")
print("=" * 65)

for i, result in enumerate(reranked_results, 1):
    print(f"\nResult: {i}")
    print("Document:", result["document_name"])
    print("Page:", result["page_number"])
    print("Rerank Score:", round(result["rerank_score"], 4))
    print("Text:", result["text"][:500])
    print("-" * 65)

CROSS-ENCODER RERANKING RESULTS

Result: 1
Document: The Consumer Protection Act, 2019.pdf
Page: 4
Rerank Score: 5.2007
Text: 4 THE GAZETTE OF INDIA EXTRAORDINARY [PART II— (iii) the right to be assured, wherever possible, access to a variety of goods, products or services at competitive prices; (iv) the right to be heard and to be assured that consumer's interests will receive due consideration at appropriate fora; (v) the right to seek redressal against unfair trade practice or restrictive trade practices or unscrupulous exploitation of consumers; and (vi) the right to consumer awareness; (10) "defect" means any faul
-----------------------------------------------------------------

Result: 2
Document: The Consumer Protection Act, 2019.pdf
Page: 3
Rerank Score: 1.6424
Text: s been made, denies or disputes the allegations contained in the complaint; (9) "consumer rights" includes,— (i) the right to be protected against the marketing of goods, products or services which are hazardous t

In [ ]:
# Step 11.1 - Build Context from Reranked Results

def build_context(results):

    context = ""

    for i, result in enumerate(results, 1):
        context += f"""
Source {i}
Document: {result['document_name']}
Page: {result['page_number']}

Content:
{result['text']}

----------------------------------------
"""

    return context


# Build context from top reranked results
retrieved_context = build_context(reranked_results)

print("Context created successfully!")
print("Total context characters:", len(retrieved_context))
print("\nCONTEXT PREVIEW\n")
print(retrieved_context[:2000])

Context created successfully!
Total context characters: 5074

CONTEXT PREVIEW


Source 1
Document: The Consumer Protection Act, 2019.pdf
Page: 4

Content:
4 THE GAZETTE OF INDIA EXTRAORDINARY [PART II— (iii) the right to be assured, wherever possible, access to a variety of goods, products or services at competitive prices; (iv) the right to be heard and to be assured that consumer's interests will receive due consideration at appropriate fora; (v) the right to seek redressal against unfair trade practice or restrictive trade practices or unscrupulous exploitation of consumers; and (vi) the right to consumer awareness; (10) "defect" means any fault, imperfection or shortcoming in the quality, quantity, potency, purity or standard which is required to be maintained by or under any law for the time being in force or under any contract, express or implied or as is claimed by the trader in any manner whatsoever in relation to any goods or product and the expression "defective" shall be con

In [ ]:
# Step 11.2 - Display source references

print("RETRIEVED SOURCE REFERENCES")
print("=" * 60)

for i, result in enumerate(reranked_results, 1):
    print(f"\nSource {i}")
    print("Document:", result["document_name"])
    print("Page:", result["page_number"])
    print("Rerank Score:", round(result["rerank_score"], 4))

print("\nTotal sources:", len(reranked_results))

RETRIEVED SOURCE REFERENCES

Source 1
Document: The Consumer Protection Act, 2019.pdf
Page: 4
Rerank Score: 5.2007

Source 2
Document: The Consumer Protection Act, 2019.pdf
Page: 3
Rerank Score: 1.6424

Source 3
Document: The Consumer Protection (Direct Selling) Rules, 2021.pdf
Page: 14
Rerank Score: 1.1676

Source 4
Document: The Consumer Protection Act, 2019.pdf
Page: 1
Rerank Score: 0.9223

Source 5
Document: The Guidelines for Prevention and Regulation of Dark Patterns, 2023.pdf
Page: 2
Rerank Score: -1.2907

Total sources: 5


In [ ]:
# Step 11.3 - Prepare RAG Prompt

def create_rag_prompt(query, context):

    prompt = f"""
You are an AI Consumer Complaint Resolution Assistant.

Answer the user's question using ONLY the information
provided in the official document context below.

Instructions:
1. Give clear and simple answers.
2. Do not invent legal information.
3. If the context does not contain the answer, say:
   "The available documents do not provide sufficient information."
4. Include document names and page numbers as sources.
5. Use bullet points when explaining multiple items.

OFFICIAL DOCUMENT CONTEXT:
{context}

USER QUESTION:
{query}

ANSWER:
"""

    return prompt


# Create prompt
rag_prompt = create_rag_prompt(
    query,
    retrieved_context
)

print("RAG prompt created successfully!")
print("Prompt length:", len(rag_prompt))
print("\nPROMPT PREVIEW:\n")
print(rag_prompt[:1500])

RAG prompt created successfully!
Prompt length: 5644

PROMPT PREVIEW:


You are an AI Consumer Complaint Resolution Assistant.

Answer the user's question using ONLY the information
provided in the official document context below.

Instructions:
1. Give clear and simple answers.
2. Do not invent legal information.
3. If the context does not contain the answer, say:
   "The available documents do not provide sufficient information."
4. Include document names and page numbers as sources.
5. Use bullet points when explaining multiple items.

OFFICIAL DOCUMENT CONTEXT:

Source 1
Document: The Consumer Protection Act, 2019.pdf
Page: 4

Content:
4 THE GAZETTE OF INDIA EXTRAORDINARY [PART II— (iii) the right to be assured, wherever possible, access to a variety of goods, products or services at competitive prices; (iv) the right to be heard and to be assured that consumer's interests will receive due consideration at appropriate fora; (v) the right to seek redressal against unfair trade pract

In [ ]:
!pip install -q openai

In [ ]:
from google.colab import userdata
from openai import OpenAI

# Retrieve API key securely
NEXUS_API_KEY = userdata.get("NEXUS_API_KEY")

# Nexus API configuration
client = OpenAI(
    api_key=NEXUS_API_KEY,
    base_url="https://nexusapi.navigatelabs.ai/v1"
)

MODEL_NAME = "gemini-2.5-flash"

print("Nexus API configuration completed!")
print("API Key: Loaded securely")
print("Base URL:", "https://nexusapi.navigatelabs.ai/v1")
print("Model:", MODEL_NAME)

Nexus API configuration completed!
API Key: Loaded securely
Base URL: https://nexusapi.navigatelabs.ai/v1
Model: gemini-2.5-flash


In [ ]:
# Step 12.3 - Test Nexus API

try:
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "user",
                "content": "Hello! Reply with: Nexus API connection successful."
            }
        ],
        temperature=0
    )

    print("NEXUS API TEST")
    print("=" * 50)
    print("Response:", response.choices[0].message.content)
    print("\nNexus API is working successfully!")

except Exception as e:
    print("Nexus API connection failed.")
    print("Error:", str(e))

Nexus API connection failed.
Error: Error code: 403 - {'error': {'message': "The requested model 'gemini-2.5-flash' is not available for this API key, or the model name is invalid. Check the models available to you and try again.", 'type': 'team_model_access_denied', 'param': 'model', 'code': '403'}}


In [ ]:
# Step 12.3 - Discover available Nexus models

try:
    models = client.models.list()

    print("AVAILABLE NEXUS MODELS")
    print("=" * 50)

    available_models = []

    for model in models.data:
        available_models.append(model.id)
        print(model.id)

    print("\nTotal available models:", len(available_models))

except Exception as e:
    print("Model discovery failed.")
    print("Error:", str(e))

AVAILABLE NEXUS MODELS
nano-banana
whisper-1
nova-micro
text-embedding-3-small
gpt-4o-mini-tts
gpt-4.1-nano

Total available models: 6


In [ ]:
# Test Nova Micro through Nexus API

MODEL_NAME = "nova-micro"

try:
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "user",
                "content": "Reply with: Nexus API connection successful."
            }
        ],
        temperature=0
    )

    print("NEXUS API TEST")
    print("=" * 50)
    print("Model:", MODEL_NAME)
    print("Response:", response.choices[0].message.content)
    print("\nNexus API is working successfully!")

except Exception as e:
    print("Model test failed.")
    print("Error:", str(e))

NEXUS API TEST
Model: nova-micro
Response: Nexus API connection successful.

Nexus API is working successfully!


In [ ]:
# Step 12.5 - RAG Answer Generation

def generate_rag_answer(prompt):

    response = client.chat.completions.create(
        model="nova-micro",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2
    )

    return response.choices[0].message.content


# Generate answer using the prepared RAG prompt
final_answer = generate_rag_answer(rag_prompt)

print("CONSUMER AI ASSISTANT - GENERATED ANSWER")
print("=" * 65)
print(final_answer)

CONSUMER AI ASSISTANT - GENERATED ANSWER
- The Consumer Protection Act, 2019 provides several rights to consumers in India. Here are the key consumer rights:

  - **Right to Safety**: Protection against marketing of hazardous goods and services.
  - **Right to be Informed**: Right to know about the quality, quantity, potency, purity, standard, and price of goods and services.
  - **Right to Choose**: Access to a variety of goods and services at competitive prices.
  - **Right to be Heard**: Right to have consumer interests considered in appropriate forums.
  - **Right to Redressal**: Right to seek redressal against unfair trade practices or exploitation.
  - **Right to Consumer Awareness**: Right to be aware of consumer rights and responsibilities.

- These rights are detailed in "The Consumer Protection Act, 2019.pdf" on page 3.


In [ ]:
#Step 13 – Answer Validation and Out-of-Scope Testing.
# Step 13.1 - Out-of-Scope Question Testing

test_query = "Who won the FIFA World Cup 2022?"

# Retrieve relevant passages
test_results = expanded_hybrid_search(
    test_query,
    top_k=10
)

# Rerank retrieved passages
test_reranked_results = cross_encoder_rerank(
    test_query,
    test_results,
    top_k=5
)

# Display retrieval results
print("OUT-OF-SCOPE QUESTION TEST")
print("=" * 65)

print("Question:", test_query)

for i, result in enumerate(test_reranked_results, 1):
    print(f"\nResult: {i}")
    print("Document:", result["document_name"])
    print("Page:", result["page_number"])
    print("Rerank Score:", round(result["rerank_score"], 4))
    print("-" * 65)

AttributeError: 'Model' object has no attribute 'encode'

In [ ]:
# Restore the original Sentence Transformer model

from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "paraphrase-multilingual-MiniLM-L12-v2"
)

print("Embedding model restored successfully!")
print("Model type:", type(model).__name__)
print("Embedding dimension:", model.get_sentence_embedding_dimension())

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding model restored successfully!
Model type: SentenceTransformer
Embedding dimension: 384


/tmp/ipykernel_475/1740060793.py:11: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", model.get_sentence_embedding_dimension())


In [ ]:
# Step 13.2 - Out-of-Scope Question Testing

test_query = "Who won the FIFA World Cup 2022?"

# Retrieve relevant passages
test_results = expanded_hybrid_search(
    test_query,
    top_k=10
)

# Rerank retrieved passages
test_reranked_results = cross_encoder_rerank(
    test_query,
    test_results,
    top_k=5
)

# Display results
print("OUT-OF-SCOPE QUESTION TEST")
print("=" * 65)
print("Question:", test_query)

for i, result in enumerate(test_reranked_results, 1):
    print(f"\nResult: {i}")
    print("Document:", result["document_name"])
    print("Page:", result["page_number"])
    print("Rerank Score:", round(result["rerank_score"], 4))
    print("-" * 65)

OUT-OF-SCOPE QUESTION TEST
Question: Who won the FIFA World Cup 2022?

Result: 1
Document: The Consumer Protection (Consumer Disputes Redressal Commissions) Amendment Rules, 2022.pdf
Page: 1
Rerank Score: -0.7701
-----------------------------------------------------------------

Result: 2
Document: The Consumer Protection (Consumer Disputes Redressal Commissions) (Amendment) Rules, 2023.pdf
Page: 1
Rerank Score: -2.6761
-----------------------------------------------------------------

Result: 3
Document: The Guidelines for Prevention of Misleading Advertisements and Endorsements for Misleading Advertisements, 2022.pdf
Page: 11
Rerank Score: -3.257
-----------------------------------------------------------------

Result: 4
Document: The Central Consumer Protection Authority (Procedure for Engagement of Experts and Professionals) Regulations, 2021.pdf
Page: 1
Rerank Score: -4.4659
-----------------------------------------------------------------

Result: 5
Document: The National Consum

In [ ]:
# Step 13.3 - Relevance-Based Answer Validation

def validate_retrieval(results, threshold=0.0):

    if not results:
        return False

    highest_score = max(
        result["rerank_score"]
        for result in results
    )

    return highest_score >= threshold


# Test the out-of-scope question
is_relevant = validate_retrieval(
    test_reranked_results,
    threshold=0.0
)

print("ANSWER VALIDATION")
print("=" * 50)
print("Question:", test_query)
print("Highest relevance score:",
      round(max(r["rerank_score"] for r in test_reranked_results), 4))

if is_relevant:
    print("Status: Relevant context found")
else:
    print("Status: Insufficient relevant information")
    print("Response: The available documents do not provide sufficient information.")

ANSWER VALIDATION
Question: Who won the FIFA World Cup 2022?
Highest relevance score: -0.7701
Status: Insufficient relevant information
Response: The available documents do not provide sufficient information.


In [ ]:
test_query = "What are the rights of consumers in India?"

test_results = expanded_hybrid_search(test_query, top_k=10)

test_reranked_results = cross_encoder_rerank(
    test_query,
    test_results,
    top_k=5
)

is_relevant = validate_retrieval(
    test_reranked_results,
    threshold=0.0
)

print("RELEVANT QUESTION VALIDATION")
print("=" * 50)
print("Question:", test_query)
print(
    "Highest relevance score:",
    round(max(r["rerank_score"] for r in test_reranked_results), 4)
)

if is_relevant:
    print("Status: Relevant context found")
    print("Response: The documents contain relevant information.")
else:
    print("Status: Insufficient relevant information")

RELEVANT QUESTION VALIDATION
Question: What are the rights of consumers in India?
Highest relevance score: 5.2007
Status: Relevant context found
Response: The documents contain relevant information.


In [ ]:
# Step 14 - Final RAG Answer Generation Test

test_query = "What are the rights of consumers in India?"

# Retrieve relevant passages
test_results = expanded_hybrid_search(
    test_query,
    top_k=10
)

# Rerank retrieved passages
test_reranked_results = cross_encoder_rerank(
    test_query,
    test_results,
    top_k=5
)

# Validate retrieved context
is_relevant = validate_retrieval(
    test_reranked_results,
    threshold=0.0
)

print("FINAL RAG ANSWER")
print("=" * 60)

if is_relevant:
    context = build_context(test_reranked_results)
    prompt = create_rag_prompt(test_query, context)
    answer = generate_rag_answer(prompt)

    print(answer)
else:
    print("Insufficient relevant information in the documents.")

FINAL RAG ANSWER
Based on the provided documents, here are the rights of consumers in India:

- **Right to Protection Against Hazardous Goods:**
  - The right to be protected against the marketing of goods that are hazardous to life and property. 
  - *Source: The Consumer Protection Act, 2019.pdf, Page: 3*

- **Right to Information:**
  - The right to be informed about the quality, quantity, potency, purity, standard, and price of goods, products, or services to protect against unfair trade practices.
  - *Source: The Consumer Protection Act, 2019.pdf, Page: 3*

- **Right to Access Competitive Prices:**
  - The right to be assured access to a variety of goods, products, or services at competitive prices.
  - *Source: The Consumer Protection Act, 2019.pdf, Page: 4*

- **Right to Seek Redressal:**
  - The right to seek redressal against unfair trade practices, restrictive trade practices, or unscrupulous exploitation of consumers.
  - *Source: The Consumer Protection Act, 2019.pdf, Page

In [ ]:
# Step 15.1 - LangGraph Agent Workflow Setup

from typing import TypedDict, List, Any
from langgraph.graph import StateGraph, START, END

print("LangGraph imported successfully!")
print("Agent workflow components are ready.")

LangGraph imported successfully!
Agent workflow components are ready.


In [ ]:
# Step 15.2 - Define Agent State

from typing import TypedDict, List, Any

class ConsumerAgentState(TypedDict):
    question: str
    retrieved_results: List[Any]
    reranked_results: List[Any]
    context: str
    is_relevant: bool
    final_answer: str

print("Consumer Agent State created successfully!")

print("\nState fields:")
for field in ConsumerAgentState.__annotations__:
    print("-", field)

Consumer Agent State created successfully!

State fields:
- question
- retrieved_results
- reranked_results
- context
- is_relevant
- final_answer


In [ ]:
# Step 15.3 - Query Analysis Agent

def query_analysis_agent(state: ConsumerAgentState):

    question = state["question"].strip()

    print("QUERY ANALYSIS AGENT")
    print("=" * 45)
    print("Original Question:", question)

    return {
        "question": question
    }

print("Query Analysis Agent created successfully!")

Query Analysis Agent created successfully!


In [ ]:
# Step 15.4 - Document Retrieval Agent

def retrieval_agent(state: ConsumerAgentState):

    question = state["question"]

    print("\nRETRIEVAL AGENT")
    print("=" * 45)

    # Retrieve relevant passages
    retrieved_results = expanded_hybrid_search(
        question,
        top_k=10
    )

    # Rerank retrieved passages
    reranked_results = cross_encoder_rerank(
        question,
        retrieved_results,
        top_k=5
    )

    print("Retrieved passages:", len(retrieved_results))
    print("Reranked passages:", len(reranked_results))

    return {
        "retrieved_results": retrieved_results,
        "reranked_results": reranked_results
    }

print("Document Retrieval Agent created successfully!")

Document Retrieval Agent created successfully!


In [ ]:
# Step 15.5 - Relevance Validation Agent

def relevance_validation_agent(state: ConsumerAgentState):

    reranked_results = state["reranked_results"]

    print("\nRELEVANCE VALIDATION AGENT")
    print("=" * 45)

    if not reranked_results:
        is_relevant = False
    else:
        is_relevant = validate_retrieval(
            reranked_results,
            threshold=0.0
        )

    print("Relevant context found:", is_relevant)

    return {
        "is_relevant": is_relevant
    }

print("Relevance Validation Agent created successfully!")

Relevance Validation Agent created successfully!


In [ ]:
# Step 15.6 - Context Building Agent

def context_building_agent(state: ConsumerAgentState):

    print("\nCONTEXT BUILDING AGENT")
    print("=" * 45)

    if state["is_relevant"]:
        context = build_context(state["reranked_results"])
    else:
        context = ""

    print("Context created:", bool(context))
    print("Context length:", len(context), "characters")

    return {
        "context": context
    }

print("Context Building Agent created successfully!")

Context Building Agent created successfully!


In [ ]:
# Step 15.7 - Response Generation Agent

def response_generation_agent(state: ConsumerAgentState):

    print("\nRESPONSE GENERATION AGENT")
    print("=" * 45)

    if not state["is_relevant"]:
        answer = (
            "Insufficient relevant information was found "
            "in the available consumer-protection documents."
        )

    else:
        prompt = create_rag_prompt(
            state["question"],
            state["context"]
        )

        answer = generate_rag_answer(prompt)

    print("Final response generated successfully!")

    return {
        "final_answer": answer
    }

print("Response Generation Agent created successfully!")

Response Generation Agent created successfully!


In [ ]:
# Step 15.8 - Build LangGraph Workflow

workflow = StateGraph(ConsumerAgentState)

# Add all five agents
workflow.add_node("query_analysis", query_analysis_agent)
workflow.add_node("retrieval", retrieval_agent)
workflow.add_node("validation", relevance_validation_agent)
workflow.add_node("context_building", context_building_agent)
workflow.add_node("response_generation", response_generation_agent)

# Define the execution order
workflow.add_edge(START, "query_analysis")
workflow.add_edge("query_analysis", "retrieval")
workflow.add_edge("retrieval", "validation")
workflow.add_edge("validation", "context_building")
workflow.add_edge("context_building", "response_generation")
workflow.add_edge("response_generation", END)

# Compile the workflow
consumer_agent = workflow.compile()

print("LANGGRAPH WORKFLOW CREATED SUCCESSFULLY!")
print("=" * 55)
print("Total agents:", 5)
print("Workflow compiled and ready.")

LANGGRAPH WORKFLOW CREATED SUCCESSFULLY!
Total agents: 5
Workflow compiled and ready.


In [ ]:
# Step 15.9 - Test Complete Agentic RAG Workflow

test_question = "What are the rights of consumers in India?"

initial_state = {
    "question": test_question,
    "retrieved_results": [],
    "reranked_results": [],
    "context": "",
    "is_relevant": False,
    "final_answer": ""
}

final_state = consumer_agent.invoke(initial_state)

print("\n" + "=" * 60)
print("FINAL AGENTIC RAG RESPONSE")
print("=" * 60)
print("Question:", test_question)
print("\nAnswer:\n")
print(final_state["final_answer"])

QUERY ANALYSIS AGENT
Original Question: What are the rights of consumers in India?

RETRIEVAL AGENT
Retrieved passages: 10
Reranked passages: 5

RELEVANCE VALIDATION AGENT
Relevant context found: True

CONTEXT BUILDING AGENT
Context created: True
Context length: 5074 characters

RESPONSE GENERATION AGENT
Final response generated successfully!

FINAL AGENTIC RAG RESPONSE
Question: What are the rights of consumers in India?

Answer:

Here are the rights of consumers in India as provided in the official documents:

- **Right to Safety**:
  - Protected against marketing of hazardous goods and services. 
  - Source: The Consumer Protection Act, 2019.pdf, Page: 3

- **Right to Information**:
  - Informed about quality, quantity, potency, purity, standard, and price of goods and services.
  - Source: The Consumer Protection Act, 2019.pdf, Page: 3

- **Right to Choose**:
  - Access to a variety of goods and services at competitive prices.
  - Source: The Consumer Protection Act, 2019.pdf, Page

In [ ]:
#  Step 16 – Context Caching  Step 16.1 - Context Caching Implementation

# Store previously retrieved information
context_cache = {}

def retrieval_agent(state: ConsumerAgentState):

    global cache_status

    question = state["question"].strip()
    cache_key = question.lower()

    print("\nRETRIEVAL AGENT")
    print("=" * 45)

    # Check whether this question is already cached
    if cache_key in context_cache:

        cache_status = "HIT"
        cached_data = context_cache[cache_key]

        print("Cache HIT: Using previously retrieved information.")

        return {
            "retrieved_results": cached_data["retrieved_results"],
            "reranked_results": cached_data["reranked_results"],
            "context": cached_data["context"]
        }

    # Perform retrieval if question is new
    cache_status = "MISS"
    print("Cache MISS: Performing fresh retrieval.")

    retrieved_results = expanded_hybrid_search(
        question,
        top_k=10
    )

    reranked_results = cross_encoder_rerank(
        question,
        retrieved_results,
        top_k=5
    )

    print("Retrieved passages:", len(retrieved_results))
    print("Reranked passages:", len(reranked_results))

    return {
        "retrieved_results": retrieved_results,
        "reranked_results": reranked_results
    }


def context_building_agent(state: ConsumerAgentState):

    print("\nCONTEXT BUILDING AGENT")
    print("=" * 45)

    # Reuse cached context if available
    if state.get("context"):
        context = state["context"]
        print("Using cached context.")

    elif state["is_relevant"]:
        context = build_context(state["reranked_results"])

    else:
        context = ""

    # Save context and retrieval results
    cache_key = state["question"].strip().lower()

    context_cache[cache_key] = {
        "retrieved_results": state["retrieved_results"],
        "reranked_results": state["reranked_results"],
        "context": context
    }

    print("Context cached successfully.")
    print("Context length:", len(context), "characters")

    return {
        "context": context
    }


cache_status = "NOT TESTED"

print("Context caching system created successfully!")

Context caching system created successfully!


In [ ]:
# Step 16.2 - Recompile Workflow with Context Caching

workflow = StateGraph(ConsumerAgentState)

workflow.add_node("query_analysis", query_analysis_agent)
workflow.add_node("retrieval", retrieval_agent)
workflow.add_node("validation", relevance_validation_agent)
workflow.add_node("context_building", context_building_agent)
workflow.add_node("response_generation", response_generation_agent)

workflow.add_edge(START, "query_analysis")
workflow.add_edge("query_analysis", "retrieval")
workflow.add_edge("retrieval", "validation")
workflow.add_edge("validation", "context_building")
workflow.add_edge("context_building", "response_generation")
workflow.add_edge("response_generation", END)

consumer_agent = workflow.compile()

print("UPDATED LANGGRAPH WORKFLOW READY!")
print("Context caching integrated successfully.")

UPDATED LANGGRAPH WORKFLOW READY!
Context caching integrated successfully.


In [ ]:
# Step 16.3 - Context Cache Testing

test_question = "What are the rights of consumers in India?"

initial_state = {
    "question": test_question,
    "retrieved_results": [],
    "reranked_results": [],
    "context": "",
    "is_relevant": False,
    "final_answer": ""
}

# First execution - expected Cache MISS
print("FIRST QUESTION EXECUTION")
print("=" * 50)

first_result = consumer_agent.invoke(initial_state)

print("\nFirst execution cache status:", cache_status)

# Second execution - expected Cache HIT
print("\n\nSECOND QUESTION EXECUTION")
print("=" * 50)

second_result = consumer_agent.invoke(initial_state)

print("\nSecond execution cache status:", cache_status)

# Final verification
print("\n" + "=" * 50)
print("CONTEXT CACHE TEST RESULTS")
print("=" * 50)
print("Cache entries:", len(context_cache))
print("First execution: MISS expected")
print("Second execution: HIT expected")
print("Cache working:", len(context_cache) > 0)

FIRST QUESTION EXECUTION


NameError: name 'consumer_agent' is not defined

In [ ]:
# Step 17.1 - Gradio Chatbot Function

def consumer_chatbot(user_question):

    if not user_question or not user_question.strip():
        return "Please enter a consumer-related question."

    initial_state = {
        "question": user_question.strip(),
        "retrieved_results": [],
        "reranked_results": [],
        "context": "",
        "is_relevant": False,
        "final_answer": ""
    }

    # Execute the complete Agentic RAG workflow
    result = consumer_agent.invoke(initial_state)

    return result["final_answer"]


print("Consumer chatbot function created successfully!")

Consumer chatbot function created successfully!


In [ ]:
# Step 17.2 - Verify Gradio Environment

import gradio as gr

print("GRADIO WEBSITE ENVIRONMENT")
print("=" * 45)

print("Gradio version:", gr.__version__)
print("Chatbot function:", callable(consumer_chatbot))
print("Agent workflow:", callable(consumer_agent.invoke))

print("\nEnvironment verification completed!")

GRADIO WEBSITE ENVIRONMENT
Gradio version: 6.31.0
Chatbot function: True
Agent workflow: True

Environment verification completed!


In [ ]:

# STEP 17.3 - COMPLETE CONSUMER CARE AI WEBSITE
# Compatible with the installed Gradio Chatbot constructor

import gradio as gr

# --------------------------------------------------
# 1. CHATBOT RESPONSE FUNCTION
# --------------------------------------------------

def respond(message, history):
    if not message or not message.strip():
        return history or [], ""

    answer = consumer_chatbot(message.strip())

    history = history or []

    updated_history = history + [
        {"role": "user", "content": message.strip()},
        {"role": "assistant", "content": answer}
    ]

    return updated_history, ""


print("Chatbot response function created successfully!")


# --------------------------------------------------
# 2. WEBSITE CSS
# --------------------------------------------------

custom_css = """
.gradio-container {
    max-width: 1150px !important;
    margin: auto !important;
}

#main-header {
    text-align: center;
    padding: 28px;
    border-radius: 15px;
    background: linear-gradient(120deg, #17365D, #2874A6);
    color: white;
    margin-bottom: 20px;
}
"""


# --------------------------------------------------
# 3. CREATE WEBSITE
# --------------------------------------------------

with gr.Blocks(title="Consumer Care AI") as demo:

    # HEADER
    gr.HTML("""
    <div id="main-header">
        <h1>Consumer Care AI</h1>
        <h3>Agentic RAG-Based Consumer Complaint Resolution Assistant</h3>
        <p>Understand your consumer rights and get guidance using official documents.</p>
    </div>
    """)

    # --------------------------------------------------
    # HOME TAB
    # --------------------------------------------------

    with gr.Tab("Home"):

        gr.Markdown("""
        ## Welcome to Consumer Care AI

        Consumer Care AI is an AI-powered assistant designed to help
        people understand consumer protection laws and complaint
        procedures in India.

        ### Main Features

        - Understand consumer rights.
        - Learn how to approach consumer complaints.
        - Ask questions through an AI-powered chatbot.
        - Receive answers with official document references.
        - Get information from government consumer-protection documents.

        ### Technologies Used

        Agentic RAG, FAISS, BM25, Cross-Encoder,
        LangGraph, Context Caching and Gradio.
        """)

    # --------------------------------------------------
    # CONSUMER RIGHTS TAB
    # --------------------------------------------------

    with gr.Tab("Consumer Rights"):

        gr.Markdown("""
        ## Six Consumer Rights in India

        ### 1. Right to Safety

        Protection against goods and services that may be hazardous
        to life and property.

        ### 2. Right to be Informed

        Access to information about quality, quantity, purity,
        standard and price.

        ### 3. Right to Choose

        Access to a variety of goods and services at competitive prices.

        ### 4. Right to be Heard

        Consumer interests should receive due consideration
        at appropriate forums.

        ### 5. Right to Seek Redressal

        Protection against unfair trade practices and the right
        to seek remedies for genuine consumer grievances.

        ### 6. Right to Consumer Awareness

        The right to acquire knowledge and skills to make informed
        consumer decisions.
        """)

    # --------------------------------------------------
    # COMPLAINT GUIDANCE TAB
    # --------------------------------------------------

    with gr.Tab("Complaint Guidance"):

        gr.Markdown("""
        ## How to Approach a Consumer Complaint

        1. Identify the product or service problem.
        2. Keep the purchase bill, receipt, warranty and other evidence.
        3. Contact the seller or service provider.
        4. Keep records of communication and responses.
        5. If the issue remains unresolved, explore official
           consumer grievance channels.

        ### Official Consumer Resources

        - [National Consumer Helpline](https://consumerhelpline.gov.in/)
        - [Department of Consumer Affairs](https://consumeraffairs.nic.in/)
        - [India Code](https://www.indiacode.nic.in/)

        *This website provides general educational guidance.
        It does not submit complaints or replace professional legal advice.*
        """)

    # --------------------------------------------------
    # AI CHATBOT TAB
    # --------------------------------------------------

    with gr.Tab("AI Chatbot"):

        gr.Markdown("""
        ## Ask Consumer Care AI

        Ask questions about consumer rights, defective products,
        refunds, unfair trade practices and complaint procedures.
        """)

        # No type parameter: use the installed default format
        chatbot = gr.Chatbot(height=430)

        message = gr.Textbox(
            placeholder="Type your consumer-related question here...",
            label="Your Question"
        )

        with gr.Row():

            send_btn = gr.Button(
                "Send",
                variant="primary"
            )

            clear_btn = gr.Button("Clear Chat")

        # Send using Enter
        message.submit(
            respond,
            inputs=[message, chatbot],
            outputs=[chatbot, message]
        )

        # Send using button
        send_btn.click(
            respond,
            inputs=[message, chatbot],
            outputs=[chatbot, message]
        )

        # Clear conversation
        clear_btn.click(
            lambda: ([], ""),
            inputs=[],
            outputs=[chatbot, message]
        )

        # Sample questions
        gr.Examples(
            examples=[
                "What are the six consumer rights in India?",
                "What can I do if I receive a defective product?",
                "How can I seek a refund?",
                "What is the purpose of the Consumer Protection Act, 2019?"
            ],
            inputs=message
        )

    # --------------------------------------------------
    # FOOTER
    # --------------------------------------------------

    gr.HTML("""
    <div style="text-align:center; padding:20px; margin-top:20px;">
        <hr>
        <h3>Consumer Care AI</h3>
        <p>Academic Capstone Project | AI Agents and RAG Systems</p>
        <p>Powered by official consumer-protection documents</p>
    </div>
    """)


# --------------------------------------------------
# 4. WEBSITE STATUS
# --------------------------------------------------

print("=" * 45)
print("CONSUMER CARE AI WEBSITE")
print("=" * 45)
print("Home Tab: Created")
print("Consumer Rights Tab: Created")
print("Complaint Guidance Tab: Created")
print("AI Chatbot Tab: Created")
print("LangGraph Agent: Connected")
print("Context Cache: Connected")
print("Website creation completed!")
print("=" * 45)


Chatbot response function created successfully!
CONSUMER CARE AI WEBSITE
Home Tab: Created
Consumer Rights Tab: Created
Complaint Guidance Tab: Created
AI Chatbot Tab: Created
LangGraph Agent: Connected
Context Cache: Connected
Website creation completed!


In [ ]:
# STEP 17.4 - LAUNCH CONSUMER CARE AI

demo.launch(
    share=True,
    debug=True,
    css=custom_css
)

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://bc5f641104392a25e2.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


KeyboardInterrupt: 

In [ ]:

# STEP 17.2.1 - RESTORE CONSUMER CHATBOT FUNCTION

def consumer_chatbot(user_question):

    if not user_question or not user_question.strip():
        return "Please enter a consumer-related question."

    initial_state = {
        "question": user_question.strip(),
        "retrieved_results": [],
        "reranked_results": [],
        "context": "",
        "is_relevant": False,
        "final_answer": ""
    }

    result = consumer_agent.invoke(initial_state)

    return result["final_answer"]


print("Consumer chatbot function restored successfully!")
print("Chatbot function available:", callable(consumer_chatbot))
print("Agent workflow available:", callable(consumer_agent.invoke))


Consumer chatbot function restored successfully!
Chatbot function available: True


NameError: name 'consumer_agent' is not defined

In [ ]:
# CHECK LANGGRAPH AGENT COMPONENTS

required_functions = [
    "query_analysis_agent",
    "retrieval_agent",
    "relevance_validation_agent",
    "context_building_agent",
    "response_generation_agent"
]

for function_name in required_functions:
    print(
        function_name,
        ":",
        callable(globals().get(function_name))
    )

print("\nWorkflow variable available:", "consumer_agent" in globals())

query_analysis_agent : False
retrieval_agent : False
relevance_validation_agent : False
context_building_agent : False
response_generation_agent : False

Workflow variable available: False


In [ ]:

# CHECK EXISTING RAG COMPONENTS

components = [
    "model",
    "faiss_index",
    "bm25",
    "chunks_df",
    "cross_encoder",
    "expanded_hybrid_search",
    "cross_encoder_rerank",
    "build_context",
    "create_rag_prompt",
    "generate_rag_answer",
    "validate_retrieval",
    "context_cache",
    "cache_status"
]

print("RAG COMPONENT VERIFICATION")
print("=" * 45)

for name in components:
    available = name in globals()
    print(f"{name}: {'AVAILABLE' if available else 'MISSING'}")

print("=" * 45)


RAG COMPONENT VERIFICATION
model: MISSING
faiss_index: MISSING
bm25: MISSING
chunks_df: MISSING
cross_encoder: MISSING
expanded_hybrid_search: MISSING
cross_encoder_rerank: MISSING
build_context: MISSING
create_rag_prompt: MISSING
generate_rag_answer: MISSING
validate_retrieval: MISSING
context_cache: MISSING
cache_status: MISSING


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

search_terms = [
    "consumer_faiss.index",
    "consumer_embeddings.npy",
    "consumer_chunks.pkl",
    "consumer_knowledge_base.csv",
    ".pdf",
    ".zip"
]

print("SEARCHING GOOGLE DRIVE")
print("=" * 50)

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    dirs[:] = [d for d in dirs if d != ".ipynb_checkpoints"]

    for filename in files:
        if any(term in filename.lower() for term in search_terms):
            print(os.path.join(root, filename))

SEARCHING GOOGLE DRIVE
/content/drive/MyDrive/GAYATHRIHSCTWO.pdf
/content/drive/MyDrive/certificate_copy.pdf
/content/drive/MyDrive/SQL_Mini_Project_Report_copy.pdf
/content/drive/MyDrive/DOC-20250502-WA0002..pdf
/content/drive/MyDrive/DOC-20250920-WA0052_250921_145857.pdf
/content/drive/MyDrive/Notes_251011_160902.pdf
/content/drive/MyDrive/photo (1).pdf
/content/drive/MyDrive/photo.pdf
/content/drive/MyDrive/DOC-20251011-WA0006_251011_163359.pdf
/content/drive/MyDrive/DOC-20251011-WA0006_251011_163427 (1).pdf
/content/drive/MyDrive/DOC-20251011-WA0006_251011_163427.pdf
/content/drive/MyDrive/Photo from Gayathri_251011_163532.pdf
/content/drive/MyDrive/aadhar (2).pdf
/content/drive/MyDrive/aadhar (1).pdf
/content/drive/MyDrive/aadhar.pdf
/content/drive/MyDrive/Black and White Clean Professional A4 Resume (6)_251011_163659.pdf
/content/drive/MyDrive/Black and White Clean Professional A4 Resume (6)_251011_163720 (1).pdf
/content/drive/MyDrive/Black and White Clean Professional A4 Resume

In [ ]:
import os

print("CHECKING FOR CONSUMER PROJECT BACKUPS")
print("=" * 50)

extensions = (".index", ".npy", ".pkl", ".csv", ".zip", ".ipynb")

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    dirs[:] = [d for d in dirs if d not in [".ipynb_checkpoints", ".trash"]]

    for filename in files:
        if filename.lower().endswith(extensions):
            if any(word in filename.lower() for word in
                   ["consumer", "faiss", "embedding", "chunk", "knowledge", "rag"]):
                print(os.path.join(root, filename))

CHECKING FOR CONSUMER PROJECT BACKUPS
/content/drive/MyDrive/Colab Notebooks/Multi_Agent_RAG_Consumer_Complaint_Assistant.ipynb
/content/drive/MyDrive/Colab Notebooks/Advanced_RAG_Government_Scheme_Explainer.ipynb
/content/drive/MyDrive/Colab Notebooks/Copy of Multi_Agent_RAG_Consumer_Complaint_Assistant.ipynb
